# EEG & EMG Practice Exercise: Solution Key

**Stationarity, Unit Root, Differencing, AR/MA/ARMA**

This notebook implements the full pipeline on **two** physiological series: EEG alpha power and EMG RMS, with EEG- and EMG-specific interpretations.

## 1. Setup and data load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox

plt.rcParams['figure.figsize'] = (10, 4)

In [ ]:
df = pd.read_csv("data/eeg_emg_ts.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.set_index("date").sort_index()
eeg = df["eeg_alpha_power"]
emg = df["emg_rms"]

print("Shape:", df.shape)
print("Date range:", df.index.min(), "to", df.index.max())
print("\nEEG alpha power - describe:")
print(eeg.describe())
print("\nEMG RMS - describe:")
print(emg.describe())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
eeg.plot(ax=axes[0], title="EEG alpha power (simulated, 1-min epochs)", ylabel="Alpha power")
emg.plot(ax=axes[1], title="EMG RMS (simulated, 1-min epochs)", ylabel="EMG RMS")
plt.tight_layout()
plt.show()
print("Interpretation: EEG shows trend and possible oscillation (alpha modulation); EMG shows trend and burst-like variability.")

## 2. Unit root tests (ADF on raw series)

In [ ]:
adf_eeg = adfuller(eeg, autolag="AIC")
adf_emg = adfuller(emg, autolag="AIC")
print("Augmented Dickey-Fuller - EEG (raw):")
print("  Test statistic:", round(adf_eeg[0], 4), "  p-value:", round(adf_eeg[1], 4))
print("Augmented Dickey-Fuller - EMG (raw):")
print("  Test statistic:", round(adf_emg[0], 4), "  p-value:", round(adf_emg[1], 4))
print("Conclusion: Both series likely non-stationary (unit root); differencing needed.")

## 3. Differencing

In [ ]:
deeg = eeg.diff().dropna()
demg = emg.diff().dropna()
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
deeg.plot(ax=axes[0], title="First difference - EEG alpha power")
demg.plot(ax=axes[1], title="First difference - EMG RMS")
plt.tight_layout()
plt.show()

In [ ]:
adf_deeg = adfuller(deeg, autolag="AIC")
adf_demg = adfuller(demg, autolag="AIC")
print("ADF - differenced EEG: p-value", round(adf_deeg[1], 4), "→ stationary.")
print("ADF - differenced EMG: p-value", round(adf_demg[1], 4), "→ stationary.")
print("Conclusion: Use d = 1 for both series.")

## 4. ACF and PACF for model selection

### 4.1 EEG (differenced)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_acf(deeg, lags=50, ax=axes[0], title="ACF - differenced EEG")
plot_pacf(deeg, lags=50, ax=axes[1], title="PACF - differenced EEG", method="ywm")
plt.tight_layout()
plt.show()
print("EEG: PACF cuts off after lag 1–2 → suggest AR(1) or AR(2) on diff → ARIMA(1,1,0) or ARIMA(2,1,0). Optional peak at period ~60 (alpha modulation).")

### 4.2 EMG (differenced)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_acf(demg, lags=50, ax=axes[0], title="ACF - differenced EMG")
plot_pacf(demg, lags=50, ax=axes[1], title="PACF - differenced EMG", method="ywm")
plt.tight_layout()
plt.show()
print("EMG: ACF cuts off after lag 1 → suggest MA(1) on diff → ARIMA(0,1,1). Typical for burst-like / short-memory EMG dynamics.")

## 5. AR/MA/ARMA fit

### 5.1 EEG - candidate models

In [ ]:
orders_eeg = [(1, 1, 0), (2, 1, 0), (1, 1, 1)]
results_eeg = {}
for order in orders_eeg:
    fit = ARIMA(eeg, order=order).fit()
    results_eeg[order] = fit
    print(f"ARIMA{order}: AIC = {fit.aic:.2f}, BIC = {fit.bic:.2f}")
best_eeg = min(results_eeg, key=lambda k: results_eeg[k].bic)
print(f"\nPreferred EEG model (BIC): ARIMA{best_eeg}")

### 5.2 EMG - candidate models

In [ ]:
orders_emg = [(0, 1, 1), (1, 1, 1), (0, 1, 2)]
results_emg = {}
for order in orders_emg:
    fit = ARIMA(emg, order=order).fit()
    results_emg[order] = fit
    print(f"ARIMA{order}: AIC = {fit.aic:.2f}, BIC = {fit.bic:.2f}")
best_emg = min(results_emg, key=lambda k: results_emg[k].bic)
print(f"\nPreferred EMG model (BIC): ARIMA{best_emg}")

In [ ]:
print("EEG preferred:")
print(results_eeg[best_eeg].summary())
print("\nEMG preferred:")
print(results_emg[best_emg].summary())

## 6. Evaluation

### 6.1 Residual diagnostics

In [ ]:
resid_eeg = results_eeg[best_eeg].resid
resid_emg = results_emg[best_emg].resid
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
plot_acf(resid_eeg, lags=40, ax=axes[0, 0], title="ACF residuals - EEG")
resid_eeg.hist(ax=axes[0, 1], bins=30, edgecolor="black", alpha=0.7); axes[0, 1].set_title("Residuals EEG")
plot_acf(resid_emg, lags=40, ax=axes[1, 0], title="ACF residuals - EMG")
resid_emg.hist(ax=axes[1, 1], bins=30, edgecolor="black", alpha=0.7); axes[1, 1].set_title("Residuals EMG")
plt.tight_layout()
plt.show()

In [ ]:
lb_eeg = acorr_ljungbox(resid_eeg, lags=20, return_df=True)
lb_emg = acorr_ljungbox(resid_emg, lags=20, return_df=True)
print("Ljung-Box EEG (first 5 lags):", lb_eeg["lb_pvalue"].head().values)
print("Ljung-Box EMG (first 5 lags):", lb_emg["lb_pvalue"].head().values)
print("If p-values > 0.05, we do not reject white noise → residuals consistent with WN.")

### 6.2 Forecast evaluation (train/test split)

In [ ]:
test_frac = 0.15
n_test = int(len(df) * test_frac)
train_eeg, test_eeg = eeg.iloc[:-n_test], eeg.iloc[-n_test:]
train_emg, test_emg = emg.iloc[:-n_test], emg.iloc[-n_test:]

fit_f_eeg = ARIMA(train_eeg, order=best_eeg).fit()
fit_f_emg = ARIMA(train_emg, order=best_emg).fit()
fc_eeg = fit_f_eeg.forecast(steps=n_test)
fc_emg = fit_f_emg.forecast(steps=n_test)

rmse_eeg = np.sqrt(np.mean((test_eeg.values - fc_eeg.values) ** 2))
rmse_emg = np.sqrt(np.mean((test_emg.values - fc_emg.values) ** 2))
mae_eeg = np.mean(np.abs(test_eeg.values - fc_eeg.values))
mae_emg = np.mean(np.abs(test_emg.values - fc_emg.values))
print("EEG - Test RMSE:", round(rmse_eeg, 4), "  MAE:", round(mae_eeg, 4))
print("EMG - Test RMSE:", round(rmse_emg, 4), "  MAE:", round(mae_emg, 4))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
train_eeg.iloc[-100:].plot(ax=axes[0], label="Train (last 100)")
test_eeg.plot(ax=axes[0], label="Actual test")
pd.Series(fc_eeg.values, index=test_eeg.index).plot(ax=axes[0], label="Forecast", linestyle="--")
axes[0].set_title("EEG alpha power - Forecast vs actual")
axes[0].legend()
train_emg.iloc[-100:].plot(ax=axes[1], label="Train (last 100)")
test_emg.plot(ax=axes[1], label="Actual test")
pd.Series(fc_emg.values, index=test_emg.index).plot(ax=axes[1], label="Forecast", linestyle="--")
axes[1].set_title("EMG RMS - Forecast vs actual")
axes[1].legend()
plt.tight_layout()
plt.show()